# Undervalued and overvalued players

Imports and paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [2]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
raw_data_dir = ROOT / "data" / "raw"
processed_data_dir = ROOT / "data" / "processed"
reports_figures = ROOT / "reports" / "figures"
tableau_data_dir = ROOT / "data" / "tableau"
tableau_data_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# The models predict the log of the market value: go back to euros before computing the error, in M€.
def mae_m_eur(y_true, y_pred):
    return mean_absolute_error(np.exp(y_true), np.exp(y_pred)) / 1e6

In this notebook, we will look at the players that are undervalued/overvalued according to the model.

For this, we need to make predictions on the whole dataset, for that we will use the cross_val_predict function to get out-of-fold predictions. In this way, we avoid overfitting and get fair predictions for every player.

A player is undervalued when the model estimates a higher value than his real market value (gap > 0), which corresponds to an overestimated player in the error analysis of notebook 05.

Loading the data

In [4]:
pl_players_2025_26 = pd.read_csv(processed_data_dir / "pl_players_2025_26.csv")
pl_players_2025_26.tail()

,player_id,name,club_id,club,position,sub_position,international_caps,age,contract_years_left,appearances,minutes_played,goals,assists,value_start_eur,value_end_eur,value_change_eur,value_change_pct
511,1111589,El Hadji Malick Diouf,379,West Ham United,Defender,Left-Back,23,21.4,4.1,32,2711,0,5,18000000.0,28000000.0,10000000.0,55.6
512,1124005,Christantus Uche,873,Crystal Palace,Attack,Centre-Forward,3,23.0,2.1,14,159,0,1,15000000.0,12000000.0,-3000000.0,-20.0
513,1193867,Pedro Lima,543,Wolverhampton Wanderers,Defender,Right-Back,1,19.9,3.1,9,259,0,0,5000000.0,4000000.0,-1000000.0,-20.0
514,1230661,Mateus Mané,543,Wolverhampton Wanderers,Midfield,Attacking Midfield,2,18.7,3.1,27,1790,3,2,200000.0,20000000.0,19800000.0,9900.0
515,1322090,Jocelin Ta Bi,289,Sunderland AFC,Attack,Right Winger,3,21.3,4.1,1,39,0,0,125000.0,2500000.0,2375000.0,1900.0


In [5]:
pl_players_2025_26.shape[0]

516

## Data preparation

Given a one season dataset (516 rows), we chose 10 folds. This ensures that the model trains on a large enough portion of the data (about 90%) in each iteration, providing robust and reliable out-of-fold predictions without excessive computation time.

In [6]:
features = ["club", "sub_position", "international_caps", "age", "contract_years_left", "minutes_played", "goals", "assists"]
model_data = pl_players_2025_26[features].copy()
model_data["value_end_log"] = np.log(pl_players_2025_26["value_end_eur"])
model_data = pd.get_dummies(model_data, columns=["sub_position", "club"], dtype=int)

In [7]:
y = model_data["value_end_log"]
X = model_data.drop(columns="value_end_log")

## Cross-validation predictions

In [8]:
kfold = KFold(n_splits=10, shuffle=True, random_state=10)
rf_model = RandomForestRegressor(random_state=10)

# Out-of-fold predictions: each player is estimated by a model trained without him.
y_predicted = cross_val_predict(rf_model, X, y, cv=kfold)

In [9]:
print("======= Random Forest Regressor (out-of-fold predictions) =======")
threshold = mae_m_eur(y, y_predicted)
print(f"Mean Absolute Error : {threshold:.1f} M€")
print(f"R² score : {r2_score(y, y_predicted):.2f}")

======= Random Forest Regressor (out-of-fold predictions) =======
Mean Absolute Error : 7.8 M€
R² score : 0.78


In [10]:
players = pl_players_2025_26.loc[X.index, ["name", "club", "position", "age", "minutes_played"]].copy()

# The real value is taken from the dataset: np.exp(np.log(value)) is not exactly the value (50 M€ would become 49.999... M€).
players["real_value_m_eur"] = pl_players_2025_26.loc[X.index, "value_end_eur"] / 1e6
players["estimated_value_m_eur"] = np.exp(y_predicted) / 1e6
players["gap_m_eur"] = players["estimated_value_m_eur"] - players["real_value_m_eur"]

conditions = [
    (players["gap_m_eur"] > threshold),
    (players["gap_m_eur"] < -threshold)
]

choices = ["undervalued", "overvalued"]
players["status"] = np.select(conditions, choices, default="fairly_valued")

players.tail().round(1)

,name,club,position,age,minutes_played,real_value_m_eur,estimated_value_m_eur,gap_m_eur,status
511,El Hadji Malick Diouf,West Ham United,Defender,21.4,2711,28.0,39.5,11.5,undervalued
512,Christantus Uche,Crystal Palace,Attack,23.0,159,12.0,4.1,-7.9,overvalued
513,Pedro Lima,Wolverhampton Wanderers,Defender,19.9,259,4.0,6.5,2.5,fairly_valued
514,Mateus Mané,Wolverhampton Wanderers,Midfield,18.7,1790,20.0,17.9,-2.1,fairly_valued
515,Jocelin Ta Bi,Sunderland AFC,Attack,21.3,39,2.5,9.7,7.2,fairly_valued


### Grouping by the status

We only keep the players with at least 900 minutes played (10 full games) in the lists: below that, their statistics are too few to judge them.

In [11]:
undervalued = players[(players["status"] == "undervalued") & (players["minutes_played"] >= 900)].sort_values("gap_m_eur", ascending=False)
overvalued = players[(players["status"] == "overvalued") & (players["minutes_played"] >= 900)].sort_values("gap_m_eur", ascending=True)
fairly_valued = players[(players["status"] == "fairly_valued") & (players["minutes_played"] >= 900)]

Profile of the undervalued and overvalued players.

In [12]:
big_six = ["Arsenal FC", "Chelsea FC", "Liverpool FC", "Manchester City", "Manchester United", "Tottenham Hotspur"]

for name, group in [("undervalued", undervalued), ("overvalued", overvalued)]:
    print(f"--- {name} : {len(group)} players ---")
    print(f"Worth 50 M€ or more : {(group['real_value_m_eur'] >= 50).sum()}")
    print(f"In a big six club : {group['club'].isin(big_six).sum()} ({group['club'].isin(big_six).mean():.0%})")
    print(f"Mean age : {group['age'].mean():.1f}")
    print(f"Median value : {group['real_value_m_eur'].median():.1f} M€")

--- undervalued : 49 players ---
Worth 50 M€ or more : 1
In a big six club : 3 (6%)
Mean age : 25.2
Median value : 20.0 M€
--- overvalued : 85 players ---
Worth 50 M€ or more : 52
In a big six club : 57 (67%)
Mean age : 26.1
Median value : 50.0 M€


#### Undervalued players

In [13]:
print(f"Number of undervalued players : {undervalued.shape[0]}")
undervalued.head(10).round(1)

Number of undervalued players : 49


,name,club,position,age,minutes_played,real_value_m_eur,estimated_value_m_eur,gap_m_eur,status
441,Dango Ouattara,Brentford FC,Attack,24.3,2282,35.0,64.1,29.1,undervalued
372,Enzo Le Fée,Sunderland AFC,Midfield,26.3,2936,28.0,53.6,25.6,undervalued
195,Brenden Aaronson,Leeds United,Midfield,25.6,2470,22.0,46.7,24.7,undervalued
488,Diego Gómez,Brighton & Hove Albion,Midfield,23.2,2139,25.0,45.6,20.6,undervalued
203,Samuel Chukwueze,Fulham FC,Attack,27.0,1074,18.0,38.0,20.0,undervalued
199,Marcus Tavernier,AFC Bournemouth,Midfield,27.2,2743,25.0,43.6,18.6,undervalued
385,David Møller Wolfe,Wolverhampton Wanderers,Defender,24.1,1048,8.0,24.3,16.3,undervalued
437,Mathys Tel,Tottenham Hotspur,Attack,21.1,1342,22.0,37.3,15.3,undervalued
411,Jake O'Brien,Everton FC,Defender,25.1,3124,18.0,33.1,15.1,undervalued
256,Jayden Bogle,Leeds United,Defender,25.9,2805,15.0,29.9,14.9,undervalued


In [14]:
print(f"Average age of the undervalued players : {undervalued["age"].mean().round(1)}")

Average age of the undervalued players : 25.2


In [15]:
undervalued_by_club = undervalued["club"].value_counts().reset_index()
undervalued_by_club.columns = ["club", "number_of_players"]
undervalued_by_club["percentage"] = (undervalued_by_club["number_of_players"] / undervalued.shape[0] * 100).round(1)
undervalued_by_club

,club,number_of_players,percentage
0,Leeds United,7,14.3
1,Wolverhampton Wanderers,6,12.2
2,Sunderland AFC,5,10.2
3,AFC Bournemouth,5,10.2
4,Brentford FC,4,8.2
5,Fulham FC,4,8.2
6,West Ham United,4,8.2
7,Brighton & Hove Albion,3,6.1
8,Burnley FC,3,6.1
9,Everton FC,2,4.1


The undervalued players are mostly regular starters of smaller clubs, around 25 years old and worth about 20 M€: only 3 of the 49 play in one of the big six clubs. For the model, their age, playing time and contract are worth more than their market value.

These players can be a lead for the recruitment, but the model also tends to overestimate the cheaper players (see notebook 05), so each name has to be checked by the scouts.

#### Overvalued players

In [16]:
print(f"Number of overvalued players : {overvalued.shape[0]}")
overvalued.head(10).round(1)

Number of overvalued players : 85


,name,club,position,age,minutes_played,real_value_m_eur,estimated_value_m_eur,gap_m_eur,status
221,Erling Haaland,Manchester City,Attack,25.9,2959,200.0,81.7,-118.3,overvalued
288,William Saliba,Arsenal FC,Defender,25.2,2615,100.0,45.6,-54.4,overvalued
169,Declan Rice,Arsenal FC,Midfield,27.4,3099,120.0,66.5,-53.5,overvalued
350,Florian Wirtz,Liverpool FC,Midfield,23.1,2390,100.0,49.3,-50.7,overvalued
241,Gabriel,Arsenal FC,Defender,28.5,2751,75.0,30.2,-44.8,overvalued
398,Moisés Caicedo,Chelsea FC,Midfield,24.6,2802,100.0,56.2,-43.8,overvalued
237,Bukayo Saka,Arsenal FC,Attack,24.7,2226,110.0,68.2,-41.8,overvalued
355,Rayan Cherki,Manchester City,Midfield,22.8,1786,90.0,49.2,-40.8,overvalued
197,Sandro Tonali,Newcastle United,Midfield,26.1,2545,80.0,39.8,-40.2,overvalued
306,Bruno Guimarães,Newcastle United,Midfield,28.5,2460,70.0,31.6,-38.4,overvalued


In [17]:
print(f"Average age of the overvalued players : {overvalued["age"].mean().round(1)}")

Average age of the overvalued players : 26.1


In [18]:
overvalued_by_club = overvalued["club"].value_counts().reset_index()
overvalued_by_club.columns = ["club", "number_of_players"]
overvalued_by_club["percentage"] = (overvalued_by_club["number_of_players"] / undervalued.shape[0] * 100).round(1)
overvalued_by_club

,club,number_of_players,percentage
0,Manchester City,15,30.6
1,Arsenal FC,12,24.5
2,Liverpool FC,10,20.4
3,Chelsea FC,8,16.3
4,Manchester United,8,16.3
5,Newcastle United,6,12.2
6,Aston Villa,5,10.2
7,AFC Bournemouth,4,8.2
8,Tottenham Hotspur,4,8.2
9,Nottingham Forest,3,6.1


In [19]:
worth_50 = overvalued[overvalued["real_value_m_eur"] >= 50]
print(len(worth_50))

52


Most of the overvalued players are stars: 52 of the 85 are worth 50 M€ or more, and two thirds play in one of the big six clubs. We already saw in notebook 05 that the model underestimates the stars, because they are rare and a random forest cannot predict beyond the values it has seen. So these players are not really overvalued by the market, it is a limit of the model.

There are also older players, such as Salah, Van Dijk, Alisson, and Bruno Fernandes. The model places great emphasis on age (we identified 41% of the model's prediction based on the player age), while the market keeps a high value for them thanks to their reputation and their past seasons, which are not in the data.

#### Fairly valued players

In [20]:
print(f"Number of fairly valued players : {fairly_valued.shape[0]}")
fairly_valued.head(10).round(1)

Number of fairly valued players : 206


,name,club,position,age,minutes_played,real_value_m_eur,estimated_value_m_eur,gap_m_eur,status
1,Casemiro,Manchester United,Midfield,34.3,2589,6.0,8.7,2.7,fairly_valued
3,Jordan Henderson,Brentford FC,Midfield,36.0,1921,1.2,0.8,-0.4,fairly_valued
4,Adam Smith,AFC Bournemouth,Defender,35.1,1080,0.3,1.8,1.5,fairly_valued
6,Danny Welbeck,Brighton & Hove Albion,Attack,35.5,2266,3.0,0.9,-2.1,fairly_valued
8,Bernd Leno,Fulham FC,Goalkeeper,34.2,3420,6.0,4.4,-1.6,fairly_valued
9,Martin Dúbravka,Burnley FC,Goalkeeper,37.4,3150,0.5,1.2,0.7,fairly_valued
11,Pascal Groß,Brighton & Hove Albion,Midfield,35.0,1637,2.5,3.1,0.6,fairly_valued
14,Kyle Walker,Burnley FC,Defender,36.0,3097,1.5,0.7,-0.8,fairly_valued
15,Kieran Trippier,Newcastle United,Defender,35.7,1592,1.5,1.1,-0.4,fairly_valued
16,Karl Darlow,Leeds United,Goalkeeper,35.7,1980,0.2,0.7,0.5,fairly_valued


## Limits

- **A gap is not a proof**: the model explains 78% of the differences in market value. The rest comes from things that are not in the data, like the talent, the reputation, the injuries or the hype. A player can look undervalued or overvalued for good reasons that the model ignores.
- **The stars**: the model always underestimates the most expensive players, so they almost all appear overvalued.  For these stars, the gap mostly comes from a limit of the model, which cannot estimate such high values.
- **The threshold**: a player is only flagged when his gap is bigger than the average error of the model (7.8 M€). A gap of 8 M€ is big for a player worth 5 M€ and small for a player worth 100 M€.
- **The data**: one season, 516 players and the Transfermarkt values only. The real transfer fees can be very different.

## CSV export

All the players are exported, with their minutes played, so that the staff can choose its own filters in Tableau.

In [ ]:
tableau_players = players.join(
    pl_players_2025_26[["sub_position", "contract_years_left", "international_caps", "goals", "assists"]]
)
tableau_players["age_group"] = pd.cut(
    tableau_players["age"], bins=[0, 21, 24, 28, 32, 50], labels=["< 21", "21-24", "25-28", "29-32", "> 32"]
)

file_path = tableau_data_dir / "undervalued_overvalued_players.csv"
tableau_players.round(1).to_csv(file_path, index=False)
print(f"{len(tableau_players)} players saved to {file_path.relative_to(ROOT)}")

516 players saved to data/tableau/undervalued_overvalued_players.csv


## Conclusion

- Each player is estimated by a random forest that never saw him (out-of-fold predictions), with an average error of 7.8 M€.
- A player is undervalued or overvalued when his gap is bigger than this error: among the players with at least 900 minutes, 49 are undervalued, 85 overvalued and 206 fairly valued.
- The undervalued players are mostly regular starters of smaller clubs, around 25 years old : they are leads for the recruitment, to be checked by the scouts.
- The overvalued players are mostly stars and older players: for them, the gap comes from the limits of the model more than from the market.